In [3]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime

BASE_URL = "https://api.mfapi.in/mf/"

# The 5 key schemes to fetch
schemes = {
    "HDFC_Top100_Direct":   125497,
    "SBI_Bluechip":         119551,
    "ICICI_Bluechip":       120503,
    "Nippon_LargeCap":      118632,
    "Axis_Bluechip":        119092,
    "Kotak_Bluechip":       120841,
}

output_dir = Path("data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

for name, code in schemes.items():
    try:
        response = requests.get(f"{BASE_URL}{code}", timeout=10)
        response.raise_for_status()
        data = response.json()

        # Parse the NAV history
        df = pd.DataFrame(data['data'])
        df['scheme_code'] = code
        df['scheme_name'] = data['meta']['scheme_name']
        df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
        df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
        df = df.sort_values('date')

        # Save to raw CSV
        file_path = output_dir / f"nav_{name}.csv"
        df.to_csv(file_path, index=False)
        print(f"✅ Saved {name}: {len(df)} rows → {file_path}")

    except Exception as e:
        print(f"❌ Failed {name} ({code}): {e}")

✅ Saved HDFC_Top100_Direct: 3092 rows → data\raw\nav_HDFC_Top100_Direct.csv
✅ Saved SBI_Bluechip: 3237 rows → data\raw\nav_SBI_Bluechip.csv
✅ Saved ICICI_Bluechip: 3308 rows → data\raw\nav_ICICI_Bluechip.csv
✅ Saved Nippon_LargeCap: 3299 rows → data\raw\nav_Nippon_LargeCap.csv
✅ Saved Axis_Bluechip: 3566 rows → data\raw\nav_Axis_Bluechip.csv
✅ Saved Kotak_Bluechip: 3302 rows → data\raw\nav_Kotak_Bluechip.csv


In [7]:
print("Unique Fund Houses:", fund_master['fund_house'].nunique())
print(fund_master['fund_house'].unique())

print("\nCategories:")
print(fund_master['category'].unique())

print("\nSub-categories:")
print(fund_master['sub_category'].unique())

print("\nRisk Categories:")
print(fund_master['risk_category'].unique())

print("\nSample AMFI Codes:")
print(fund_master['amfi_code'].head(10).tolist())

print("\nAMFI Code Data Type:")
print(fund_master['amfi_code'].dtype)

Unique Fund Houses: 10
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

Categories:
['Equity' 'Debt']

Sub-categories:
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

Risk Categories:
['Moderate' 'Very High' 'Low' 'High' 'Moderately High']

Sample AMFI Codes:
[119551, 119552, 119598, 119599, 119120, 100016, 125497, 100033, 125498, 100025]

AMFI Code Data Type:
int64


In [6]:
fund_master.columns.tolist()

['amfi_code',
 'fund_house',
 'scheme_name',
 'category',
 'sub_category',
 'plan',
 'launch_date',
 'benchmark',
 'expense_ratio_pct',
 'exit_load_pct',
 'min_sip_amount',
 'min_lumpsum_amount',
 'fund_manager',
 'risk_category',
 'sebi_category_code']

In [8]:
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")

print(nav_history.columns.tolist())

['amfi_code', 'date', 'nav']


In [10]:
# Get all AMFI codes from both datasets

master_codes = set(fund_master['amfi_code'].astype(str))

nav_codes = set(nav_history['amfi_code'].astype(str))

# Find codes present in fund_master but missing in nav_history

missing_in_nav = master_codes - nav_codes

print(f"Total codes in fund_master: {len(master_codes)}")
print(f"Total codes in nav_history: {len(nav_codes)}")

print(f"Codes in master but NOT in nav_history: {len(missing_in_nav)}")

if missing_in_nav:
    print("\nMissing Codes:")
    print(missing_in_nav)
else:
    print("\nAll AMFI codes are present in nav_history.")

Total codes in fund_master: 40
Total codes in nav_history: 40
Codes in master but NOT in nav_history: 0

All AMFI codes are present in nav_history.


In [11]:
print("----- DATA QUALITY SUMMARY -----")

print("\nFund Master Null Values:")
print(fund_master.isnull().sum())

print("\nNAV History Null Values:")
print(nav_history.isnull().sum())

print("\nFund Master Duplicate Rows:")
print(fund_master.duplicated().sum())

print("\nNAV History Duplicate Rows:")
print(nav_history.duplicated().sum())

----- DATA QUALITY SUMMARY -----

Fund Master Null Values:
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

NAV History Null Values:
amfi_code    0
date         0
nav          0
dtype: int64

Fund Master Duplicate Rows:
0

NAV History Duplicate Rows:
0


# Day 1 Findings

## Fund Master Dataset
- Total Fund Houses: 10
- Categories: Equity, Debt
- Risk Categories: Moderate, High, Very High, Moderately High, Low

## AMFI Validation
- Validated AMFI codes between fund_master and nav_history.
- Checked for missing scheme codes.

## Data Quality Checks
- Missing value analysis completed.
- Duplicate record analysis completed.
- Dataset structure verified.

## Conclusion
All datasets were successfully loaded and explored. Initial data quality validation was completed.